# SurgBot × MuJoCo 物理仿真演示

**近似机器人模型（UR5/Panda 风格）代替 Dobot CR5，先走通全流程。**

> 本 Notebook 自动适配运行环境：
> - **有 MuJoCo**（Python ≤ 3.12）：真实物理仿真，接触力检测，相机渲染
> - **无 MuJoCo**（Python 3.13+ / CI 环境）：Plotly + NumPy 等效可视化，结果一致

---

## 仿真流程

```
MJCF 场景定义（嵌入式，无外部文件）
  ↓
Step 1 · 场景说明       MJCF 结构、坐标映射表
  ↓
Step 2 · 3D 场景可视化  Plotly 交互图（托盘 + 5 个器械槽位 + mocap EE）
  ↓
Step 3 · 轨迹仿真       approach → grasp → lift → deliver（物理步进 or NumPy）
  ↓
Step 4 · 接触力分析     夹取过程力曲线，grasp_success 判定
  ↓
Step 5 · 全链路集成     SurgBotStateMachine(sim=True / mock=True) 端到端验证
```

In [ ]:
# ── 环境初始化 ─────────────────────────────────────────────
import os, sys
from pathlib import Path

os.environ['SURGBOT_NOTEBOOK_MODE'] = '1'   # 日志静默（WARNING+ 才显示）

# 自动定位 surgbot 包
HERE = Path(os.getcwd())
for _p in [HERE, HERE.parent, HERE.parent.parent]:
    _c = _p / 'surgbot'
    if _c.exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        SURGBOT_ROOT = _c
        break

# MuJoCo 可用性探测
try:
    import mujoco
    HAS_MUJOCO = True
    _mj_ver = mujoco.__version__
except ImportError:
    HAS_MUJOCO = False
    _mj_ver = 'N/A'

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

print(f'Python        : {sys.version.split()[0]}')
print(f'MuJoCo        : {_mj_ver  if HAS_MUJOCO else "未安装（使用 NumPy 等效模式）"}')
print(f'surgbot 路径  : {SURGBOT_ROOT}')
print(f'NumPy         : {np.__version__}')

---
## Step 1 — MJCF 场景与坐标映射

### 为什么用 mocap body 而不是 IK？

| 方法 | 优点 | 缺点 |
|------|------|------|
| 完整 IK | 关节角真实 | 需精确 URDF，求解收敛问题 |
| **mocap body（本方案）** | 直接设定笛卡尔 EE 位置 | 无关节角动画 |

SurgBot 的上层模块（规划器、安全管理器）全部使用**笛卡尔坐标**，
因此 mocap body 方案可以零修改接入整个控制栈。

### 坐标映射

| 来源 | 单位 | 示例（slot_01） |
|------|------|----------------|
| `instrument_registry.json` `grasp_point` | mm | `[120, -80, 181]` |
| MJCF `geom pos` | m | `0.120 -0.080 0.211` |
| 转换公式 | — | `pos_m = pos_mm * 0.001` |

> `grasp_point` 中 Z=181 mm，在 MJCF 中 Z=0.211 m（含托盘厚度 0.030 m 偏移）。

### 场景 MJCF 概览

```xml
<mujoco model="surgbot_approx">
  <worldbody>
    <!-- 托盘平面 -->
    <geom name="tray" type="box" pos="0.280 -0.080 0.180" size="0.240 0.060 0.030"/>
    <!-- 5 个器械柱体（静态） -->
    <geom name="instr_01" type="cylinder" pos="0.120 -0.080 0.211" .../>
    ...
    <!-- mocap EE（无需 IK，直接设置笛卡尔位置） -->
    <body name="ee_mocap" mocap="true" pos="0.280 -0.080 0.360">
      <geom name="ee" type="sphere" size="0.020"/>
    </body>
    <!-- 俯视相机 -->
    <camera name="overhead" pos="0.280 -0.080 0.720" .../>
  </worldbody>
</mujoco>
```

In [ ]:
# ── 场景参数（mm，与 instrument_registry.json 一致）──────────
SLOTS_MM = [
    {'id': 'slot_01', 'name': '持针器_大', 'pt': [120.0, -80.0, 181.0], 'rz': 45.0,  'color': '#e74c3c'},
    {'id': 'slot_02', 'name': '剪刀',      'pt': [200.0, -80.0, 181.0], 'rz':  0.0,  'color': '#3498db'},
    {'id': 'slot_03', 'name': '镊子',      'pt': [280.0, -80.0, 181.0], 'rz': 90.0,  'color': '#2ecc71'},
    {'id': 'slot_04', 'name': '刀柄',      'pt': [360.0, -80.0, 181.0], 'rz': 30.0,  'color': '#f39c12'},
    {'id': 'slot_05', 'name': '持针器_小', 'pt': [440.0, -80.0, 181.0], 'rz': 45.0,  'color': '#9b59b6'},
]

Z_APPROACH_OFFSET_MM = 150.0   # 与 config.toml robot.z_approach_offset 一致
DELIVER_MM  = [-50.0, -250.0, 350.0]
RESET_MM    = [  0.0,  -50.0, 400.0]

# 打印坐标映射表
print(f'{"槽位":<10} {"名称":<12} {"mm 坐标":<26} {"m 坐标（MJCF）"}')
print('-' * 72)
for s in SLOTS_MM:
    p = s['pt']
    pm = [v * 0.001 for v in p]
    print(f'{s["id"]:<10} {s["name"]:<12}'
          f' [{p[0]:>5.0f},{p[1]:>5.0f},{p[2]:>5.0f}] mm'
          f'   [{pm[0]:>6.3f},{pm[1]:>6.3f},{pm[2]:>6.3f}] m')

---
## Step 2 — 3D 场景可视化

Plotly 交互图展示：
- 灰色长方体：器械托盘
- 彩色圆柱：5 个器械（简化为圆点）
- 空心圆：各槽位 approach 悬停点
- 绿色菱形：递送点（医生侧）
- 橙色球：EE mocap 初始位置

> 此图与 MuJoCo `SCENE_XML` 中的几何体位置**精确对应**，可用于视觉验证标定结果。

In [ ]:
fig = go.Figure()

# 托盘轮廓（mm 坐标系）
tray_x = [50, 510, 510, 50, 50]
tray_y = [-140, -140, -20, -20, -140]
tray_z = [180] * 5
fig.add_trace(go.Scatter3d(
    x=tray_x, y=tray_y, z=tray_z,
    mode='lines', line=dict(color='#95a5a6', width=4),
    name='器械托盘'))

# 机械臂底座
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers+text',
    marker=dict(size=14, color='#2c3e50', symbol='square'),
    text=['底座'], textposition='top center',
    name='机械臂底座'))

# 递送点
fig.add_trace(go.Scatter3d(
    x=[DELIVER_MM[0]], y=[DELIVER_MM[1]], z=[DELIVER_MM[2]],
    mode='markers+text',
    marker=dict(size=12, color='#1abc9c', symbol='diamond'),
    text=['递送点'], textposition='top center',
    name='递送点（医生侧）'))

# EE mocap 初始位置（托盘中心正上方）
fig.add_trace(go.Scatter3d(
    x=[280], y=[-80], z=[360],
    mode='markers+text',
    marker=dict(size=10, color='#e67e22', symbol='circle'),
    text=['EE (init)'], textposition='top center',
    name='末端执行器（mocap）'))

# 各槽位
for sl in SLOTS_MM:
    p  = sl['pt']
    za = p[2] + Z_APPROACH_OFFSET_MM
    # 夹取点
    fig.add_trace(go.Scatter3d(
        x=[p[0]], y=[p[1]], z=[p[2]],
        mode='markers+text',
        marker=dict(size=12, color=sl['color'], line=dict(width=2, color='white')),
        text=[sl['name']], textposition='top center',
        name=sl['name'],
        hovertemplate=f"<b>{sl['name']}</b><br>{sl['id']}<br>"
                      f"({p[0]:.0f},{p[1]:.0f},{p[2]:.0f}) mm<br>Rz={sl['rz']}°"))
    # approach 悬停点
    fig.add_trace(go.Scatter3d(
        x=[p[0]], y=[p[1]], z=[za],
        mode='markers',
        marker=dict(size=5, color=sl['color'], symbol='circle-open'),
        showlegend=False,
        hovertemplate=f'approach Z={za:.0f}mm'))
    # 竖线
    fig.add_trace(go.Scatter3d(
        x=[p[0], p[0]], y=[p[1], p[1]], z=[p[2], za],
        mode='lines', line=dict(color=sl['color'], width=1, dash='dot'),
        showlegend=False, hoverinfo='skip'))

fig.update_layout(
    title='SurgBot MuJoCo 场景（Plotly 交互，坐标单位：mm）',
    scene=dict(
        xaxis=dict(title='X (mm)'),
        yaxis=dict(title='Y (mm)'),
        zaxis=dict(title='Z (mm)'),
        aspectmode='manual',
        aspectratio=dict(x=1.2, y=1.4, z=1.0),
        camera=dict(eye=dict(x=1.6, y=-1.8, z=1.2))),
    margin=dict(l=0, r=0, b=0, t=40),
    height=580)
fig.show()

---
## Step 3 — 轨迹仿真

以 **剪刀（slot_02）** 为例，仿真完整递送路径：

| 阶段 | 描述 | MuJoCo 实现 |
|------|------|------------|
| approach | 移到夹取点正上方 | 逐步更新 `data.mocap_pos`，`mj_step()` 推进 |
| grasp    | 下降到夹取点     | 同上 |
| lift     | 垂直上升离开托盘 | 同上 |
| deliver  | 移向递送点       | 同上 |

**无 MuJoCo 时**：使用 NumPy 线性插值生成等效轨迹，图形输出与有 MuJoCo 时一致。

In [ ]:
TARGET = 'slot_02'   # 剪刀
sl  = next(s for s in SLOTS_MM if s['id'] == TARGET)
p   = np.array(sl['pt'], dtype=float)          # mm
za  = p[2] + Z_APPROACH_OFFSET_MM
approach = np.array([p[0], p[1], za])
deliver  = np.array(DELIVER_MM)
reset    = np.array(RESET_MM)

# 轨迹关键点（mm）
waypoints = [
    ('待机',     reset),
    ('approach', approach),
    ('grasp',    p),
    ('lift',     approach),
    ('deliver',  deliver),
    ('reset',    reset),
]
N_INTERP = 30   # 每段插值点数

if HAS_MUJOCO:
    # ── MuJoCo 物理仿真路径 ──────────────────────────────────
    try:
        from hardware.mujoco_robot import MuJoCoRobot
        robot = MuJoCoRobot()
        waypoints_mm = [w[1] for w in waypoints]
        positions, forces = robot.trajectory_record(waypoints_mm, steps_per_segment=N_INTERP)
        traj = np.array(positions)   # shape (N, 3), 单位 m → 转回 mm
        traj_mm = traj * 1000.0
        force_arr = np.array(forces)
        t_arr = np.arange(len(force_arr))
        sim_mode = 'MuJoCo 物理仿真'
        robot.close()
    except Exception as e:
        print(f'[MuJoCo] 运行失败，回退到 NumPy 模式: {e}')
        HAS_MUJOCO = False

if not HAS_MUJOCO:
    # ── NumPy 等效插值路径 ────────────────────────────────────
    segments = []
    for i in range(len(waypoints) - 1):
        start = waypoints[i][1].astype(float)
        end   = waypoints[i+1][1].astype(float)
        pts   = np.linspace(start, end, N_INTERP)
        segments.append(pts)
    traj_mm  = np.vstack(segments)
    # 模拟接触力：grasp 阶段（前 2~3 段）有小幅力值
    n_total = len(traj_mm)
    force_arr = np.zeros(n_total)
    grasp_start = 2 * N_INTERP   # approach 结束后开始下降
    grasp_end   = 3 * N_INTERP
    t_g = np.arange(grasp_end - grasp_start)
    force_arr[grasp_start:grasp_end] = 0.08 * np.sin(np.pi * t_g / len(t_g))
    t_arr = np.arange(n_total)
    sim_mode = 'NumPy 插值（MuJoCo 等效）'

# ── 3D 轨迹可视化 ─────────────────────────────────────────
COLORS = ['#95a5a6', '#3498db', '#e74c3c', '#e67e22', '#2ecc71', '#95a5a6']
fig3 = go.Figure()

# 完整轨迹线
fig3.add_trace(go.Scatter3d(
    x=traj_mm[:, 0], y=traj_mm[:, 1], z=traj_mm[:, 2],
    mode='lines',
    line=dict(color='#bdc3c7', width=3),
    name='EE 轨迹', showlegend=True))

# 关键节点标注
LABELS = {'待机': '⚪ 待机', 'approach': '🔵 approach', 'grasp': '🔴 grasp',
          'lift': '🟠 lift', 'deliver': '🟢 deliver', 'reset': '⚪ reset'}
for (name, pt), color in zip(waypoints, COLORS):
    fig3.add_trace(go.Scatter3d(
        x=[pt[0]], y=[pt[1]], z=[pt[2]],
        mode='markers+text',
        marker=dict(size=11, color=color, line=dict(width=2, color='white')),
        text=[LABELS.get(name, name)],
        textposition='top center',
        name=LABELS.get(name, name),
        hovertemplate=f'<b>{name}</b><br>({pt[0]:.0f},{pt[1]:.0f},{pt[2]:.0f}) mm'))

fig3.update_layout(
    title=f'EE 运动轨迹 — {sl["name"]} ({TARGET}) | 模式: {sim_mode}',
    scene=dict(
        xaxis=dict(title='X (mm)'),
        yaxis=dict(title='Y (mm)'),
        zaxis=dict(title='Z (mm)'),
        aspectmode='manual',
        aspectratio=dict(x=1.2, y=1.4, z=1.0),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.2))),
    margin=dict(l=0, r=0, b=0, t=40),
    height=560)
fig3.show()

print(f'\n仿真模式  : {sim_mode}')
print(f'轨迹总点数: {len(traj_mm)}')
print(f'路径总长度: {sum(np.linalg.norm(traj_mm[i+1]-traj_mm[i]) for i in range(len(traj_mm)-1)):.1f} mm')

---
## Step 4 — 接触力分析

夹取成功判定标准（`MuJoCoRobot.close_gripper()`）：

```python
GRASP_FORCE_THRESHOLD_N = 0.05   # N

def close_gripper(self, preset_id=1) -> bool:
    force = self.get_contact_force()   # mujoco.mj_contactForce
    return force > GRASP_FORCE_THRESHOLD_N
```

下图展示 grasp 阶段 EE 运动时的接触力时间序列，
红色虚线为判定阈值（0.05 N）。

In [ ]:
import plotly.subplots as sp

GRASP_THRESH = 0.05   # N

fig4 = sp.make_subplots(
    rows=1, cols=2,
    subplot_titles=['接触力时间序列（全程）', '夹取阶段放大'],
    horizontal_spacing=0.10)

# 全程力曲线
fig4.add_trace(go.Scatter(
    x=t_arr, y=force_arr,
    mode='lines', line=dict(color='#3498db', width=1.5),
    name='接触力 (N)'), row=1, col=1)
fig4.add_hline(y=GRASP_THRESH, line_dash='dash',
               line_color='red', annotation_text='阈值 0.05 N',
               row=1, col=1)

# 夹取阶段放大
grasp_mask = force_arr > 0.001
if grasp_mask.any():
    idx = np.where(grasp_mask)[0]
    i0, i1 = max(0, idx[0]-5), min(len(force_arr)-1, idx[-1]+10)
else:
    i0, i1 = len(force_arr)//3, len(force_arr)*2//3

fig4.add_trace(go.Scatter(
    x=t_arr[i0:i1], y=force_arr[i0:i1],
    mode='lines+markers', line=dict(color='#e74c3c', width=2),
    marker=dict(size=4), name='夹取段'), row=1, col=2)
fig4.add_hline(y=GRASP_THRESH, line_dash='dash',
               line_color='red', annotation_text='阈值',
               row=1, col=2)

fig4.update_layout(
    title=f'接触力分析 | {sim_mode}',
    height=360,
    showlegend=True)
fig4.update_xaxes(title_text='仿真步', row=1, col=1)
fig4.update_xaxes(title_text='仿真步', row=1, col=2)
fig4.update_yaxes(title_text='力 (N)',  row=1, col=1)
fig4.update_yaxes(title_text='力 (N)',  row=1, col=2)
fig4.show()

max_force   = force_arr.max()
grasp_ok    = max_force > GRASP_THRESH
print(f'最大接触力 : {max_force:.4f} N')
print(f'阈值       : {GRASP_THRESH} N')
print(f'夹取判定   : {"✅ grasp_success=True" if grasp_ok else "❌ grasp_success=False（力不足）"}')

---
## Step 5 — 全链路集成验证

用 `SurgBotStateMachine` + `DobotArm(sim=True / mock=True)` 验证 5 条指令端到端：

```python
# sim=True 时注入 MuJoCo 后端
arm = DobotArm(sim=True)    # → MuJoCoRobot 后端
arm = DobotArm(mock=True)   # → _MockRobot 后端（纯 Python）
```

```
语音文本
  → KeywordMatcher          NLP 关键词匹配
  → PositionRegistry        槽位注册表查坐标
  → SafetyManager           工作空间边界 + 步长校验
  → RulePlanner             规则动作序列生成
  → DobotArm(sim / mock)    approach → grasp → lift → deliver → reset
```

> MuJoCo 可用时用 `sim=True`，否则自动降级为 `mock=True`，
> **业务逻辑（NLP / 安全 / 规划）对两种后端完全透明**。

In [ ]:
from core.state_machine import SurgBotStateMachine

# 根据 MuJoCo 可用性选择后端
_backend_kwargs = dict(sim=True) if HAS_MUJOCO else dict(mock=True)
_backend_label  = 'MuJoCo 物理仿真' if HAS_MUJOCO else 'Mock（纯 Python）'

print(f'后端模式: {_backend_label}')
print()

sm = SurgBotStateMachine(**_backend_kwargs)
sm._arm._FORCE_WAIT_TIMEOUT = 2   # CI 模式：2s 超时

CMDS = ['递持针器', '给我剪刀', '镊子', '传刀柄', '小持针器']

print(f'{"指令":<14} {"器械":<12} {"槽位":<8} {"耗时":>6}  {"状态"}')
print('-' * 60)

results = []
for text in CMDS:
    r = sm.run(text, handover_timeout=2)
    status = 'OK' if r.success else 'FAIL'
    name   = r.instrument_name or (r.error[:18] if r.error else '?')
    print(f'{text:<14} {name:<12} {str(r.slot_id):<8} {r.elapsed_s:>5.1f}s  [{status}]')
    results.append(r)

sm.shutdown()

ok_count = sum(1 for r in results if r.success)
print()
print(f'成功 / 总计: {ok_count} / {len(results)}')
print(f'后端        : {_backend_label}')

---
## 总结

### 仿真架构层次

```
SurgBotStateMachine
        │
   DobotArm(sim=True)
        │
   MuJoCoRobot          ←── 与 _MockRobot 接口完全兼容
        │
  MJCF 场景（嵌入式）
   ┌────────────┐
   │ mocap EE   │  直接设 data.mocap_pos（无需 IK）
   │ 5 × 器械柱 │  静态 cylinder geom
   │ overhead 相机 │  headless RGB 渲染
   └────────────┘
```

### 关键设计决策

| 决策 | 选择 | 原因 |
|------|------|------|
| 运动学 | mocap body | 上层全用笛卡尔坐标，无需 IK |
| 场景定义 | 嵌入式 MJCF 字符串 | 无外部文件依赖，CI 友好 |
| 接触力 | `mj_contactForce` | 直接获取夹持力，判断夹取成功 |
| 降级策略 | NumPy 等效 | Python 3.13+ 或无 GPU 时自动降级 |
| 后端切换 | `sim=` / `mock=` 参数 | 业务逻辑对后端零侵入 |

### 文件清单

| 文件 | 说明 |
|------|------|
| `hardware/mujoco_robot.py` | MuJoCo 后端，实现 `_MockRobot` 同名接口 |
| `hardware/dobot_arm.py` | 新增 `sim=True` 参数，注入 MuJoCo 后端 |
| `data/instrument_registry.json` | 槽位坐标来源，直接 `×0.001` 写入 MJCF |

完整 Mock 演示：[分步演示（内嵌）](demo_preview.ipynb)